本程式使用pyautogui來完成一個blitz的遊戲機器人

In [35]:
import pyautogui as pyg
pyg.sleep(2)
basex=615
basey=509
gap=60

In [36]:
def gem_position(r, c):
    """回傳第 r 列、第 c 欄寶石中心點的 (x, y) 座標，索引範圍為 0 到 7。"""
    if not (0 <= r < 8 and 0 <= c < 8):
        raise ValueError("r 和 c 必須介於 0 到 7")
    return (basex + c * gap, basey + r * gap)

In [37]:
def gem_position(r,c):
    x=basex+c*gap
    y=basey+r*gap
    return(x,y)


In [38]:
print(gem_position(0,0))
print(gem_position(0,1))
print(gem_position(1,0))

(615, 509)
(675, 509)
(615, 569)


In [39]:
for i in range(8):
    x,y=gem_position(0,i)
    pyg.click(x,y,duration=1)

In [40]:
def get_gem_color(r,c,im):
    position=gem_position(r,c)
    return im.getpixel(position)

In [41]:
def can_move(r, c, im):
    """判斷第 r 列、第 c 欄的寶石往左交換後是否能形成三連。"""
    if not (0 <= r < 8 and 0 <= c < 8):
        raise ValueError("r 和 c 必須介於 0 到 7")
    if c == 0:
        return False

    colors = [[get_gem_color(row, col, im) for col in range(8)]
              for row in range(8)]
    if colors[r][c] == colors[r][c - 1]:
        return False

    # 模擬和左側寶石交換，不實際操作滑鼠。
    colors[r][c], colors[r][c - 1] = colors[r][c - 1], colors[r][c]

    def has_match_at(row, col):
        color = colors[row][col]
        for dr, dc in ((0, 1), (1, 0)):
            count = 1
            for direction in (-1, 1):
                nr = row + dr * direction
                nc = col + dc * direction
                while (0 <= nr < 8 and 0 <= nc < 8
                       and colors[nr][nc] == color):
                    count += 1
                    nr += dr * direction
                    nc += dc * direction
            if count >= 3:
                return True
        return False

    # 交換後新形成的三連必定經過其中一個交換位置。
    return has_match_at(r, c) or has_match_at(r, c - 1)

In [42]:
im=pyg.screenshot()
print(can_move_left(2,2,im))

False


In [48]:
import time
import pyautogui as pyg

# 1. 存下目前時間
start_time = time.time()
timeout = 65  # 設定總執行時間（例如 60 秒，可依需求調整）

# 2. 螢幕截圖
im = pyg.screenshot()

# 3. 進入無窮迴圈
while True:
    # 檢查是否超過時間（避免無限跑下去）
    if time.time() - start_time > timeout:
        print("時間到，結束執行。")
        break

    moved = False

    # 4. 對於畫面中的每個寶石 (r, c)
    for r in range(8):
        for c in range(8):
            # 內層也檢查超時
            if time.time() - start_time > timeout:
                break

            direction = None

            # 5. 判斷是否可以往右、左、上、下移動
            if can_move_right(r, c, im):
                direction = (0, 1)      # 右
            elif can_move_left(r, c, im):
                direction = (0, -1)     # 左
            elif can_move_up(r, c, im):
                direction = (-1, 0)     # 上
            elif can_move_down(r, c, im):
                direction = (1, 0)      # 下

            # 6. 若其中一個可以，就將寶石 (r, c) 移到對應方向
            if direction is not None:
                x, y = gem_position(r, c)
                dr, dc = direction
                
                # 執行滑鼠拖曳
                pyg.moveTo(x, y)
                pyg.dragTo(x + dc * gap, y + dr * gap, duration=0.15)
                
                moved = True
                
                # 移動後稍微停頓，等待遊戲消除與掉落動畫
                time.sleep(0.25)
                
                # 7. 再次截圖（更新盤面）
                im = pyg.screenshot()
                break  # 找到並移動一顆後，跳出內層迴圈重新掃描新盤面

        if moved or (time.time() - start_time > timeout):
            break

    # 如果這一輪掃完整個棋盤都沒有找到可以動的寶石，稍微休息一下避免 CPU 飆高
    if not moved:
        time.sleep(0.1)
        im = pyg.screenshot()  # 沒動也更新一次截圖

時間到，結束執行。
